# P Abhilash Goud — Customer Churn Prediction

Customer Churn Prediction project using RFM features and Logistic Regression.

## Workflow
1. Load cleaned transaction data.
2. Use 14-May-2026 as the historical cutoff.
3. Create customer-level RFM features.
4. Define churn from future purchasing behaviour.
5. Train Logistic Regression.
6. Evaluate accuracy, precision, recall and confusion matrix.
7. Generate churn probability and customer risk levels.

Risk levels:
- Low Risk: probability < 40%
- Medium Risk: 40% to < 70%
- High Risk: >= 70%


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

DATA_PATH = "data/cleaned_customer_transactions.csv"
CUTOFF_DATE = pd.Timestamp("2026-05-14")

df = pd.read_csv(DATA_PATH, parse_dates=["Order_Date"])
print("Rows:", len(df))
print("Unique customers:", df["Customer_ID"].nunique())


In [ ]:
historical = df[df["Order_Date"] <= CUTOFF_DATE].copy()
future = df[df["Order_Date"] > CUTOFF_DATE].copy()

customer = (
    historical.groupby("Customer_ID")
    .agg(
        Order_Count=("Order_ID", "count"),
        Total_Revenue=("Revenue", "sum"),
        Avg_Order_Value=("Revenue", "mean"),
        First_Purchase=("Order_Date", "min"),
        Last_Purchase=("Order_Date", "max"),
    )
    .reset_index()
)

customer["Recency"] = (CUTOFF_DATE - customer["Last_Purchase"]).dt.days
customer["Frequency"] = customer["Order_Count"]
customer["Monetary"] = customer["Total_Revenue"]

future_customers = set(future["Customer_ID"])
customer["Churn_Status"] = (~customer["Customer_ID"].isin(future_customers)).astype(int)

print(customer["Churn_Status"].value_counts())


In [ ]:
features = ["Recency", "Frequency", "Monetary", "Avg_Order_Value"]
X = customer[features]
y = customer["Churn_Status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy :", f"{accuracy_score(y_test, y_pred):.2%}")
print("Precision:", f"{precision_score(y_test, y_pred):.2%}")
print("Recall   :", f"{recall_score(y_test, y_pred):.2%}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=1000, random_state=42))
])
final_model.fit(X, y)

customer["Churn_Probability"] = final_model.predict_proba(X)[:, 1]
customer["Risk_Level"] = np.select(
    [
        customer["Churn_Probability"] >= 0.70,
        customer["Churn_Probability"] >= 0.40
    ],
    ["High Risk", "Medium Risk"],
    default="Low Risk"
)

risk_table = customer.sort_values("Churn_Probability", ascending=False)

print(risk_table["Risk_Level"].value_counts())
risk_table.head(20)


## Interpretation

The model produces a predicted churn probability and risk level for each customer. These results represent predicted risk and associations in the available data, not proof of the causes of churn.
